# 13. API Testing Notebook

Comprehensive tests for the system's internal APIs and components.

In [1]:
import sys, os
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")

test_results = {"passed": 0, "failed": 0, "details": []}

def run_test(name, func):
    try:
        if func() in [None, True]:
            test_results["passed"] += 1
            print(f"✅ {name}")
            return True
    except Exception as e:
        test_results["details"].append({"name": name, "error": str(e)})
    test_results["failed"] += 1
    print(f"❌ {name}")
    return False

Working directory: D:\University\Uni\Semester 7\Generative AI\Project\Cirq-RAG-Code-Assistant


In [2]:
print("=" * 50)
print("📋 CONFIG & IMPORT TESTS")
print("=" * 50)

run_test("Config Import", lambda: __import__('src.cirq_rag_code_assistant.config', fromlist=['get_config']))
run_test("RAG Imports", lambda: all([__import__('src.rag.embeddings'), __import__('src.rag.retriever')]))
run_test("Agent Imports", lambda: all([__import__('src.agents.designer'), __import__('src.agents.validator')]))
run_test("Evaluation Imports", lambda: __import__('src.evaluation.metrics'))
run_test("CLI Imports", lambda: __import__('src.cli.main'))

📋 CONFIG & IMPORT TESTS
❌ Config Import
✅ RAG Imports


2025-12-07 17:22:13 | INFO     | src.cirq_rag_code_assistant.config.logging:setup_all:138 | Logging configuration completed


✅ Agent Imports
❌ Evaluation Imports
❌ CLI Imports


False

In [3]:
print("\n" + "=" * 50)
print("🔧 COMPONENT TESTS")
print("=" * 50)

from src.rag.embeddings import EmbeddingModel
from src.rag.vector_store import VectorStore
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever
from src.rag.generator import Generator

run_test("EmbeddingModel", lambda: EmbeddingModel().get_embedding_dimension() > 0)

em = EmbeddingModel()
run_test("VectorStore", lambda: VectorStore(em.get_embedding_dimension()) is not None)

vs = VectorStore(em.get_embedding_dimension())
kb = KnowledgeBase(embedding_model=em, vector_store=vs)
run_test("KnowledgeBase", lambda: kb is not None)

# Load KB
try:
    kb.load_from_directory()
    kb.load_index()
    print(f"   Loaded {len(kb.entries)} entries")
except: pass

run_test("Retriever", lambda: Retriever(kb) is not None)
run_test("Generator", lambda: Generator(Retriever(kb)) is not None)

2025-12-07 17:22:13 | INFO     | config.config_loader:load:93 | ✅ Loaded configuration from D:\University\Uni\Semester 7\Generative AI\Project\Cirq-RAG-Code-Assistant\config\config.json
2025-12-07 17:22:14 | INFO     | src.rag.embeddings:__init__:97 | Loading embedding model: BAAI/bge-base-en-v1.5
2025-12-07 17:22:14 | INFO     | src.rag.embeddings:__init__:98 | Using device: cpu



🔧 COMPONENT TESTS
2025-12-07 17:22:14,002 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:227 - Load pretrained SentenceTransformer: BAAI/bge-base-en-v1.5


2025-12-07 17:22:18 | INFO     | src.rag.embeddings:__init__:106 | ✅ Embedding model loaded successfully
2025-12-07 17:22:18 | INFO     | src.rag.embeddings:__init__:113 | Embedding dimension: 768
2025-12-07 17:22:18 | INFO     | src.rag.embeddings:__init__:97 | Loading embedding model: BAAI/bge-base-en-v1.5
2025-12-07 17:22:18 | INFO     | src.rag.embeddings:__init__:98 | Using device: cpu


✅ EmbeddingModel
2025-12-07 17:22:18,187 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:227 - Load pretrained SentenceTransformer: BAAI/bge-base-en-v1.5


2025-12-07 17:22:22 | INFO     | src.rag.embeddings:__init__:106 | ✅ Embedding model loaded successfully
2025-12-07 17:22:22 | INFO     | src.rag.embeddings:__init__:113 | Embedding dimension: 768
2025-12-07 17:22:22 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2025-12-07 17:22:22 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2025-12-07 17:22:22 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 768
2025-12-07 17:22:22 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2025-12-07 17:22:22 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2025-12-07 17:22:22 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 768
2025-12-07 17:22:22 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2025-12-07 17:22:22 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading k

✅ VectorStore
✅ KnowledgeBase
   Loaded 140 entries
✅ Retriever
✅ Generator


True

In [4]:
print("\n" + "=" * 50)
print("🤖 AGENT TESTS")
print("=" * 50)

from src.agents.designer import DesignerAgent
from src.agents.optimizer import OptimizerAgent  
from src.agents.validator import ValidatorAgent
from src.agents.educational import EducationalAgent

retriever = Retriever(kb)
generator = Generator(retriever)

run_test("DesignerAgent", lambda: DesignerAgent(retriever, generator) is not None)
run_test("OptimizerAgent", lambda: OptimizerAgent(retriever=retriever) is not None)
run_test("ValidatorAgent", lambda: ValidatorAgent(retriever=retriever) is not None)
run_test("EducationalAgent", lambda: EducationalAgent(retriever) is not None)

2025-12-07 17:22:22 | INFO     | src.rag.retriever:__init__:170 | Initialized Retriever with top_k=5, threshold=0.7, topic_boost=0.15, hybrid_scoring=True
2025-12-07 17:22:22 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/cirq-designer-agent
2025-12-07 17:22:22 | INFO     | src.agents.base_agent:__init__:78 | Initialized DesignerAgent agent
2025-12-07 17:22:22 | INFO     | src.agents.base_agent:__init__:78 | Initialized OptimizerAgent agent
2025-12-07 17:22:22 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/cirq-optimizer-agent
2025-12-07 17:22:22 | INFO     | src.agents.optimizer:__init__:74 | OptimizerAgent initialized (RAG: enabled)
2025-12-07 17:22:22 | INFO     | src.agents.base_agent:__init__:78 | Initialized ValidatorAgent agent
2025-12-07 17:22:22 | INFO     | src.tools.simulator:__init__:82 | Initialized simulator simulator
2025-12-07 17:22:22 | INFO     | src.agents.validator:__init__:94 | ValidatorAgent initial


🤖 AGENT TESTS
✅ DesignerAgent
✅ OptimizerAgent
✅ ValidatorAgent
✅ EducationalAgent


True

In [5]:
print("\n" + "=" * 50)
print("🎼 ORCHESTRATOR & VALIDATION TESTS")
print("=" * 50)

from src.orchestration.orchestrator import Orchestrator

designer = DesignerAgent(retriever, generator)
optimizer = OptimizerAgent(retriever=retriever)
validator = ValidatorAgent(retriever=retriever)
educational = EducationalAgent(retriever)

run_test("Orchestrator Init", lambda: Orchestrator(
    designer=designer, optimizer=optimizer, 
    validator=validator, educational=educational) is not None)

# Test validator with valid code
valid_code = "import cirq\nq = cirq.LineQubit.range(2)\nc = cirq.Circuit(cirq.H(q[0]))"
run_test("Validator (valid code)", lambda: validator.run({"code": valid_code}).get('validation_passed', False))

# Test validator with invalid code  
run_test("Validator (invalid code)", lambda: not validator.run({"code": "@@invalid@@"}).get('validation_passed', True))

2025-12-07 17:22:22 | INFO     | src.agents.base_agent:__init__:78 | Initialized DesignerAgent agent
2025-12-07 17:22:22 | INFO     | src.agents.base_agent:__init__:78 | Initialized OptimizerAgent agent
2025-12-07 17:22:22 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/cirq-optimizer-agent
2025-12-07 17:22:22 | INFO     | src.agents.optimizer:__init__:74 | OptimizerAgent initialized (RAG: enabled)
2025-12-07 17:22:22 | INFO     | src.agents.base_agent:__init__:78 | Initialized ValidatorAgent agent
2025-12-07 17:22:22 | INFO     | src.tools.simulator:__init__:82 | Initialized simulator simulator
2025-12-07 17:22:22 | INFO     | src.agents.validator:__init__:94 | ValidatorAgent initialized in 'local' mode (RAG: enabled)
2025-12-07 17:22:22 | INFO     | src.agents.base_agent:__init__:78 | Initialized EducationalAgent agent
2025-12-07 17:22:22 | INFO     | src.agents.educational:__init__:239 | EducationalAgent using config from: agents.educational.model
202


🎼 ORCHESTRATOR & VALIDATION TESTS
✅ Orchestrator Init


2025-12-07 17:22:31 | INFO     | src.agents.validator:execute:164 | 💡 Using LLM-fixed code for retry...
2025-12-07 17:22:31 | INFO     | src.agents.validator:execute:123 | 🔄 Re-validating fixed code (attempt 2/3)...
2025-12-07 17:22:31 | INFO     | src.tools.simulator:simulate:147 | Simulation completed. Histogram: {'0': 1024}
2025-12-07 17:22:32 | INFO     | src.agents.validator:execute:155 | ✅ Validation passed on attempt 2
2025-12-07 17:22:32 | INFO     | src.agents.validator:execute:121 | Validating code (attempt 1/3)...
2025-12-07 17:22:32 | INFO     | src.agents.validator:_execute_local:463 | Running LLM analysis for compilation error fixing...
2025-12-07 17:22:32 | ERROR    | src.agents.validator:_execute_local:577 | ValidatorAgent local execution error: 'dict' object has no attribute 'lower'
2025-12-07 17:22:32 | WARNING  | src.agents.validator:execute:167 | ⚠️ No LLM fix available, retrying with same code...
2025-12-07 17:22:32 | INFO     | src.agents.validator:execute:123 | 🔄

✅ Validator (valid code)
❌ Validator (invalid code)


False

In [6]:
print("\n" + "=" * 50)
print("📊 TEST SUMMARY")
print("=" * 50)

total = test_results["passed"] + test_results["failed"]
rate = test_results["passed"] / total if total > 0 else 0

print(f"\nTotal: {total} | Passed: {test_results['passed']} | Failed: {test_results['failed']}")
print(f"Pass Rate: {rate:.1%}")

if test_results["details"]:
    print("\nFailed tests:")
    for d in test_results["details"]:
        print(f"  - {d['name']}: {d.get('error', 'Unknown')}")


📊 TEST SUMMARY

Total: 17 | Passed: 13 | Failed: 4
Pass Rate: 76.5%
